# Lab 5: Filesystem Tool Security — PathSanitizer & Intent-Based Permissions

**Module 05 — Security & Guardrails**

Originally lived as a bonus in module 02's tool-calling lab. It belongs here: path traversal and intent-based file-access controls are first-class guardrails, not an aside on tool calling.

## Objectives

1. Use `os.path.realpath` (not `abspath`) to defeat both directory traversal *and* symlink escape
2. Layer **intent-based permissions** (read / write / append) on top of path sanitisation
3. Wrap both checks in a tool that **never raises** into the agent loop — every failure becomes a structured `{success, error}` response

## Prerequisite

You should already know the structured-tool-return contract from module 02 Lab 1: `{"success": bool, "result": Any, "error": str | None}`. This lab reuses that contract for filesystem tools.

In [ ]:
import os
from enum import Enum
from typing import Dict, Any

print('Setup OK')

---
## Part 1: The Threat Model

A naive filesystem tool gives the LLM the ability to read any file your process can. A confused or malicious caller can exploit three classes of bug:

| # | Attack | Example |
|---|--------|---------|
| 1 | **Directory traversal** | `path="../../etc/passwd"` — relative `..` segments escape your base directory |
| 2 | **Symlink escape** | A symlink inside the base directory whose target is `/etc/passwd` |
| 3 | **Permission mismatch** | A tool advertised as "read-only" being asked to open a file in write mode |

`os.path.abspath` defeats #1 but **not #2** — it does not resolve symlinks. `os.path.realpath` defeats both.

---
## Part 2: `PathSanitizer` — Resolve, Then Check Containment

The check is two lines:

1. Resolve the candidate path to an absolute, **symlink-followed** path (`os.path.realpath`).
2. Verify it sits inside the base directory's resolved path.

If it escapes, raise `SecurityError`. The caller catches it and returns a structured error — no exception leaks.

In [ ]:
class SecurityError(Exception):
    """Raised when a path or permission check fails."""
    pass


class PathSanitizer:
    """Validates file paths to prevent directory traversal AND symlink escape."""

    @staticmethod
    def validate_safe_path(base_dir: str, target_path: str) -> str:
        # realpath() resolves '..' segments AND follows symlinks.
        # abspath() only does the former — a symlink inside base_dir pointing outside
        # would bypass an abspath-only check.
        abs_base   = os.path.realpath(base_dir)
        abs_target = os.path.realpath(os.path.join(base_dir, target_path))

        # On case-insensitive filesystems (Windows, macOS default APFS),
        # normalize the case before comparing so 'C:\\Users\\foo' matches 'c:\\users\\foo'.
        norm_base   = os.path.normcase(abs_base)
        norm_target = os.path.normcase(abs_target)

        if not norm_target.startswith(norm_base + os.sep) and norm_target != norm_base:
            raise SecurityError(
                f"Path traversal blocked: '{target_path}' escapes the allowed directory."
            )
        return abs_target

---
## Part 3: `FilePermission` — Intent-Based Access

A tool that only needs to **read** should never be callable in write mode, and vice versa. We model this with an enum each tool declares upfront, and a `SecureFileAccess` wrapper that enforces it at `open()` time.

In [ ]:
class FilePermission(Enum):
    READ   = "read"
    WRITE  = "write"
    APPEND = "append"


class SecureFileAccess:
    """Combines PathSanitizer (no traversal/symlink escape) with FilePermission (intent-based access)."""

    _ALLOWED_MODES = {
        FilePermission.READ:   {"r", "rb"},
        FilePermission.WRITE:  {"w", "wb", "x"},
        FilePermission.APPEND: {"a", "ab"},
    }

    def __init__(self, base_dir: str, permission: FilePermission):
        self.base_dir   = base_dir
        self.permission = permission

    def open(self, relative_path: str, mode: str = "r"):
        safe_path = PathSanitizer.validate_safe_path(self.base_dir, relative_path)
        allowed   = self._ALLOWED_MODES[self.permission]
        if mode not in allowed:
            raise PermissionError(
                f"Mode '{mode}' is not allowed for {self.permission.value} permission. Allowed: {allowed}"
            )
        return open(safe_path, mode, encoding="utf-8" if "b" not in mode else None)

---
## Part 4: Tools That Return Structured Errors

Every tool returns `{"success": bool, "result": Any, "error": str | None}`. `SecurityError` and `PermissionError` are caught inside the tool and converted — the agent loop never sees a raw exception.

In [ ]:
BASE_DIR = "."

def list_files(path: str = ".") -> Dict[str, Any]:
    """Lists files in a directory, blocked from escaping BASE_DIR."""
    try:
        safe_path = PathSanitizer.validate_safe_path(BASE_DIR, path)
        return {"success": True, "result": os.listdir(safe_path), "error": None}
    except SecurityError as e:
        return {"success": False, "result": None, "error": f"Security: {e}"}
    except FileNotFoundError:
        return {"success": False, "result": None, "error": f"Directory not found: '{path}'"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}

def read_file(path: str) -> Dict[str, Any]:
    """Read a file — read permission only, traversal/symlink blocked."""
    fs = SecureFileAccess(base_dir=BASE_DIR, permission=FilePermission.READ)
    try:
        with fs.open(path, "r") as f:
            return {"success": True, "result": f.read(), "error": None}
    except SecurityError as e:
        return {"success": False, "result": None, "error": f"Security: {e}"}
    except PermissionError as e:
        return {"success": False, "result": None, "error": f"Permission: {e}"}
    except FileNotFoundError:
        return {"success": False, "result": None, "error": f"File not found: '{path}'"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}

def write_file(path: str, content: str) -> Dict[str, Any]:
    """Write a file — write permission only, traversal/symlink blocked."""
    fs = SecureFileAccess(base_dir=BASE_DIR, permission=FilePermission.WRITE)
    try:
        safe_path = PathSanitizer.validate_safe_path(BASE_DIR, path)
        os.makedirs(os.path.dirname(safe_path) or ".", exist_ok=True)
        with fs.open(path, "w") as f:
            f.write(content)
        return {"success": True, "result": f"Wrote {len(content)} chars to '{path}'.", "error": None}
    except SecurityError as e:
        return {"success": False, "result": None, "error": f"Security: {e}"}
    except PermissionError as e:
        return {"success": False, "result": None, "error": f"Permission: {e}"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}

print("Tools defined: list_files, read_file, write_file")

---
## Part 5: Test the Boundaries

Each call below should either succeed cleanly or be blocked with a structured error — **no raised exception** reaches the caller.

In [ ]:
print("1. Write a file inside BASE_DIR:")
print(write_file("test_output.txt", "Hello from the agent!"))

print("\n2. Read it back:")
print(read_file("test_output.txt"))

print("\n3. Path traversal attempt on read_file ('../../etc/passwd'):")
print(read_file("../../etc/passwd"))

print("\n4. Path traversal attempt on write_file ('../../evil.txt'):")
print(write_file("../../evil.txt", "pwned"))

print("\n5. Mode mismatch — opening a read-only handle in write mode (direct API):")
try:
    ro = SecureFileAccess(base_dir=BASE_DIR, permission=FilePermission.READ)
    ro.open("test_output.txt", mode="w")
except PermissionError as e:
    print(f"   Blocked: {e}")

if os.path.exists("test_output.txt"):
    os.remove("test_output.txt")
    print("\nCleanup: test_output.txt removed.")

---
## Exercise: Verify the Symlink Defence

`os.path.abspath` would let a symlink inside `BASE_DIR` whose target is `/etc` (or `C:\\Windows` on Windows) silently bypass the check. `os.path.realpath` does not.

Manually verify this on a Unix-like system:

```bash
ln -s /etc ./pwn_link    # create a symlink inside BASE_DIR pointing outside
```

Then run:

```python
print(read_file("pwn_link/passwd"))
```

With `realpath` (above), this should be blocked. If you temporarily swap in `abspath`, it will succeed — confirming the threat is real and the fix matters.

Don't forget: `rm pwn_link` when you're done.

---
## Summary: Two Layers of Defence

| Layer | Blocks |
|-------|--------|
| `PathSanitizer` + `realpath` | Directory traversal (`../`) **and** symlink escape |
| `FilePermission` | Mode mismatch — a read tool can never be called in write mode |

Both checks run *before* the filesystem is touched. Failures are caught inside the tool and returned as `{"success": False, ...}` — the agent loop never sees a raw exception, the conversation continues, and the user gets a clear explanation.

**This is the same defence-in-depth pattern from module 05 Sessions 1–3, applied to filesystem tools.**